[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Reading What They Learned &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 test rows


The worked examples' two models, rebuilt here from the same made-up stations so that each answer runs on
its own.

In [2]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], train["tmin"])
xgboost_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=0)
xgboost_model.fit(train[FEATURES], train["tmin"])
booster = xgboost_model.get_booster()
lightgbm_contributions = lightgbm_model.predict(test[FEATURES], pred_contrib=True)

**1.** Print LightGBM's importances by gain as a share of their total, sorted from the largest, beside its split counts as a share of theirs.

In [3]:
splits = pd.Series(lightgbm_model.feature_importances_, index=lightgbm_model.feature_name_)
gain = pd.Series(lightgbm_model.booster_.feature_importance("gain"), index=lightgbm_model.feature_name_)
shares = pd.DataFrame({"gain share": gain / gain.sum(), "split share": splits / splits.sum()})
print(shares.sort_values("gain share", ascending=False, kind="stable").round(3))

             gain share  split share
day_of_year       0.686        0.198
snow_cm           0.113        0.074
latitude          0.105        0.160
elevation_m       0.030        0.148
cloud_cover       0.027        0.131
wind_ms           0.020        0.125
coast_km          0.011        0.126
terrain           0.007        0.038


By gain, `day_of_year` takes 0.686 of the total and `snow_cm` 0.113, and by splits they take 0.198 and 0.074. `coast_km` makes 12.6% of the splits for 1.1% of the gain.

**2.** Rank the features by XGBoost's `weight` and by its `total_gain`, and print the two rankings side by side.

In [4]:
weight = pd.Series(booster.get_score(importance_type="weight"))
total_gain = pd.Series(booster.get_score(importance_type="total_gain"))
print(pd.DataFrame({
    "by weight": weight.sort_values(ascending=False, kind="stable").index,
    "by total_gain": total_gain.sort_values(ascending=False, kind="stable").index,
}))

     by weight by total_gain
0  day_of_year   day_of_year
1      wind_ms       snow_cm
2  cloud_cover      latitude
3     latitude   elevation_m
4  elevation_m   cloud_cover
5     coast_km       wind_ms
6      snow_cm      coast_km
7      terrain       terrain


The two rankings agree on the first feature and the last, `day_of_year` and `terrain`, and disagree in between. `wind_ms` is second by weight and sixth by total gain, and `snow_cm` is second by total gain and seventh by weight.

**3.** Find the test day the XGBoost model predicts warmest, and print each feature's contribution to that prediction from XGBoost's booster.

In [5]:
warmest = int(np.argmax(xgboost_model.predict(test[FEATURES])))
day = xgb.DMatrix(test[FEATURES].iloc[[warmest]], enable_categorical=True)
parts = pd.Series(booster.predict(day, pred_contribs=True)[0], index=FEATURES + ["average prediction"])
print(test[FEATURES].iloc[[warmest]].to_string(index=False))
print(parts.sort_values(ascending=False, kind="stable").round(2).to_string())

 latitude  elevation_m  coast_km terrain  day_of_year  cloud_cover  wind_ms  snow_cm
     58.6          364     123.0   fjord          203         0.92      2.0      0.0
day_of_year           7.02
latitude              3.49
snow_cm               1.18
cloud_cover           1.05
coast_km              0.52
terrain              -0.01
wind_ms              -0.19
elevation_m          -0.22
average prediction   -1.52


The warmest prediction is for a fjord station at latitude 58.6, 364 meters up and 123 kilometers from the coast, on day 203, late in July, under an overcast sky. The season contributes +7.02 degrees, the southern latitude +3.49, no snow on the ground, +1.18, and the cloud +1.05.

**4.** Average `wind_ms`'s LightGBM contribution over test days grouped by the worked examples' `sky` groups and by whether the wind is below 2 meters a second, and say what the difference means.

In [6]:
sky = pd.cut(test["cloud_cover"], [-0.01, 0.2, 0.8, 1.0], labels=["clear", "partly cloudy", "overcast"])
wind = pd.Series(lightgbm_contributions[:, FEATURES.index("wind_ms")], index=test.index)
calm = test["wind_ms"] < 2
print(wind.groupby([sky, calm.rename("calm")], observed=True).mean().round(2))

cloud_cover    calm 
clear          False    0.43
               True    -1.41
partly cloudy  False    0.33
               True    -0.94
overcast       False    0.17
               True    -0.54
dtype: float64


On calm days, with wind below 2 meters a second, the wind's contribution is −1.41 degrees under a clear sky and −0.54 under an overcast one, and on windier days it is above 0 under every sky. Calm air cools a night most when the sky is clear, another interaction the made-up stations were built with.

**5.** Train an `XGBClassifier` for frost (`tmin < 0`) with 200 trees and `random_state=0` on the training stations, take the contributions of the first test day from its booster, and turn their sum into a probability with `1 / (1 + np.exp(-total))` to compare with `predict_proba`.

In [7]:
frost = (train["tmin"] < 0).astype(int)
frost_xgboost = xgb.XGBClassifier(n_estimators=200, random_state=0).fit(train[FEATURES], frost)
first_day = test[FEATURES].iloc[[0]]
day = xgb.DMatrix(first_day, enable_categorical=True)
parts = pd.Series(frost_xgboost.get_booster().predict(day, pred_contribs=True)[0], index=FEATURES + ["bias"])
print(parts.round(3).to_string())
total = float(parts.sum())
print("sum:", round(total, 3), "| from the contributions:", round(1 / (1 + np.exp(-total)), 3),
      "| predict_proba:", round(float(frost_xgboost.predict_proba(first_day)[0, 1]), 3))

latitude      -3.552
elevation_m   -1.189
coast_km       0.166
terrain       -0.916
day_of_year    2.697
cloud_cover    0.341
wind_ms        0.432
snow_cm       -0.827
bias           0.230
sum: -2.617 | from the contributions: 0.068 | predict_proba: 0.068


On these made-up stations, XGBoost's contributions for the first test day add up to −2.617 in log-odds, and the logistic function turns that sum into 0.068, the same as `predict_proba`. Frost is the positive class, so a negative contribution pushes toward no frost, and this station's latitude, −3.552, pushes hardest, against +2.697 from the day of the year.

---

&#8592; **Back to:** [Reading What They Learned](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/13-reading-what-they-learned.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
